# Centroid comparison: Ballet (eloy) vs photutils 2D Gaussian vs astropy Moffat

Compare the CNN centroids that `bandaid` currently uses (`eloy.centroid.ballet_centroid`)
against two analytic centroiders applied to the **same bayer-balanced image** at the
**same input pixel positions** (Gaia coords projected through a per-image WCS).

**Two correctness constraints enforced throughout:**

1. **Symmetric in-frame gating.** All three methods are run on exactly the same subset of
   inputs — the `in_frame` mask, computed once before any centroiding. Off-frame Gaia
   entries are skipped uniformly (otherwise `eloy.ballet_centroid` would still produce a
   'centroid' on a cutout silently padded with `np.median(data)`).
2. **Box consistency.** A fit on a `PHOTUTILS_BOX × PHOTUTILS_BOX` cutout can only
   physically yield a centroid within `box/2` pixels of the input. `TRFLSQFitter` (Moffat)
   and `centroid_2dg` are unbounded by default, so on faint cutouts they can return a
   center far outside the cutout that `Cutout2D.to_original_position` then translates
   back into image coords. Every method is post-validated by `enforce_box_consistency`,
   and the Moffat fit gets explicit parameter bounds.

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.modeling.models import Moffat2D, Const2D
from astropy.modeling.fitting import TRFLSQFitter
from astropy.nddata import Cutout2D
from astropy.stats import median_absolute_deviation
from astropy.table import Table
from photutils.centroids import centroid_sources, centroid_2dg
from twirl import compute_wcs, gaia_radecs
from tqdm.auto import tqdm

from eloy.centroid import Ballet, ballet_centroid
from bandaid import (
    ReferenceData,
    bayer_balance_image,
    calibration_sequence,
    neighbor_contamination_flag,
    prepare_image,
)
from bandaid.photometry import THRESH

%matplotlib inline

IMAGE = Path(
    '/Users/mattcraig/Dropbox/MSUM/Research/photometry-transform-stuff/eloy/stwg/'
    'photometry_raw_data_t_cr_bor/Light_T CrB_10.0s_IRCUT_20250818-203439.fit'
)
USER_META = Path(
    '/Users/mattcraig/Dropbox/MSUM/Research/photometry-transform-stuff/eloy/stwg/mark_umd.json'
)
GAIA_MAG_LIMIT = 15
PHOTUTILS_BOX = 15  # match Ballet's 15x15 cutout for apples-to-apples comparison


## 1. Reference setup

In [ ]:
ref_data, ref_metadata, ref_coords, ref_fwhm, _ = calibration_sequence(IMAGE, threshold=THRESH)
print(f'detected {len(ref_coords)} sources, FWHM = {ref_fwhm:.2f} px')


## 2. Gaia query, WCS solve, contamination filter

In [ ]:
fov = ref_metadata['fov_rad']
center = SkyCoord.from_name(ref_metadata['object'])

all_radecs, mags = gaia_radecs(center, 2 * fov, magnitude=True)
good_gaia = mags <= GAIA_MAG_LIMIT
all_radecs = all_radecs[good_gaia]
mags = mags[good_gaia]
print(f'Gaia stars after mag cut: {len(mags)}')

wcs = compute_wcs(ref_coords[:15], all_radecs[:15], tolerance=1)
gaia_xy = np.array(
    wcs.world_to_pixel(SkyCoord(all_radecs[:, 0], all_radecs[:, 1], unit='deg'))
).T

contaminated = neighbor_contamination_flag(gaia_xy, mags, ref_fwhm,
                                           tolerance=0.01, beta=3)
useful_gaia_radec = all_radecs[~contaminated]
useful_gaia_mags = mags[~contaminated]
print(f'dropped {contaminated.sum()} contaminated; {len(useful_gaia_radec)} kept')

ref = ReferenceData.from_pixel_coords(ref_coords, wcs, useful_gaia_radec, Ballet())


## 3. Run `prepare_image` (production-style) and capture the inputs

In [ ]:
user_specific_metadata = json.loads(USER_META.read_text())
gaia_photo_coords = SkyCoord(
    useful_gaia_radec[:, 0], useful_gaia_radec[:, 1], unit='deg',
)
# Reuse the WCS we already solved (used to compute gaia_xy for the contamination
# filter) so contamination geometry and aligned_coords share one projection.
img = prepare_image(
    IMAGE, ref,
    detect_on_bayer_balanced=True,
    photometry_coords=gaia_photo_coords,
    user_specific_metadata=user_specific_metadata,
    wcs=wcs,
)
ballet_xy_prod = np.asarray(img.centroid_coords)  # has silent input-fallback on NaN
input_xy       = np.asarray(img.aligned_coords)
fwhm           = float(img.fwhm)
print(f'image FWHM = {fwhm:.2f} px,  inputs = {input_xy.shape}')

# Sanity: gaia_xy and input_xy should now agree exactly for stars that survived
# the contamination filter, since both came from the same WCS.
useful_idx_in_all = np.flatnonzero(~contaminated)
assert np.allclose(gaia_xy[useful_idx_in_all], input_xy, equal_nan=True), \
    'gaia_xy and input_xy disagree even after pinning to one WCS'


## 4. Reproduce the bayer-balanced image

In [ ]:
balanced = img.calibrated_data.copy()
bayer_balance_image(balanced)
# bayer_balance_image is deterministic, so this reproduces the working_image
# prepare_image used internally. Sanity check: at least one pixel changed.
assert np.any(balanced != img.calibrated_data), \
    'balanced image is identical to the input — balancing did not run'
print(f'balanced shape={balanced.shape}, mean={balanced.mean():.2f}, std={balanced.std():.2f}')


## 5. Box-consistency helper

Used after every centroid method. NaNs out any centroid that lies more than `box/2`
pixels from its input — physically impossible for a fit confined to that cutout.

In [ ]:
def enforce_box_consistency(centroid_xy, input_xy, box):
    half = box / 2.0
    finite = np.isfinite(centroid_xy).all(axis=1)
    dxy = centroid_xy - input_xy
    bad = finite & (np.abs(dxy) > half).any(axis=1)
    cleaned = centroid_xy.copy()
    cleaned[bad] = np.nan
    return cleaned, int(bad.sum())


## 6. In-image mask — gates **all three** centroiding methods

Computed *before* any centroiding so every method sees the same gated input. This matters
for Ballet too: `eloy.ballet_centroid` calls `utils.cutout(..., fill_value=np.median(data))`,
so for a star whose 15×15 footprint is *partially* off the chip the cutout silently
gets median-filled (and for a star fully off the chip, `eloy.utils.cutout` swallows
`NoOverlapError` and substitutes a zero array). Either way the CNN happily emits a
'centroid' on a fake cutout. The `in_frame` gate avoids that entirely.

In [ ]:
edge = PHOTUTILS_BOX // 2 + 2
ny, nx = balanced.shape
in_frame = (
    np.isfinite(input_xy).all(axis=1)
    & (input_xy[:, 0] > edge) & (input_xy[:, 0] < nx - edge)
    & (input_xy[:, 1] > edge) & (input_xy[:, 1] < ny - edge)
)
print(f'in-frame stars: {in_frame.sum()} / {len(in_frame)}')


## 7. Re-run Ballet with `nans=True` (gated on in_frame), then box-consistency

In [ ]:
ballet_inframe = np.asarray(
    ballet_centroid(balanced, input_xy[in_frame], ref.cnn, nans=True)
)
ballet_raw = np.full_like(input_xy, np.nan, dtype=float)
ballet_raw[in_frame] = ballet_inframe
n_silent = int(np.sum(np.isnan(ballet_inframe).any(axis=1)))
ballet_xy, n_ballet_outbox = enforce_box_consistency(ballet_raw, input_xy, PHOTUTILS_BOX)
ballet_ok = np.isfinite(ballet_xy).all(axis=1)
print(f'Ballet NaN from CNN     : {n_silent} '
      f'(of {in_frame.sum()} in-frame attempts)')
print(f'Ballet outside-box drop : {n_ballet_outbox}')
print(f'ballet_ok               : {ballet_ok.sum()} / {in_frame.sum()} attempted')


## 8. photutils 2D Gaussian centroids (gated on in_frame, post-validated)

In [ ]:
def centroid_2dg_sky_sub(data, mask=None):
    # photutils centroid_2dg fits a bare Gaussian2D with no constant background term.
    # Its docstring says the cutout should be background-subtracted; without that, the
    # Gaussian centroid is biased by the sky pedestal. Median per cutout matches what
    # the Moffat's Const2D term and Ballet's CNN effectively provide.
    # centroid_sources requires a `mask` keyword on centroid_func — and if a mask is
    # provided we must exclude masked pixels from the median too, otherwise the
    # pedestal estimate is biased by the very pixels we're trying to ignore.
    bkg_sample = data if mask is None else np.where(mask, np.nan, data)
    bkg = np.nanmedian(bkg_sample)
    return centroid_2dg(data - bkg, mask=mask)


with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    gx, gy = centroid_sources(
        balanced,
        input_xy[in_frame, 0], input_xy[in_frame, 1],
        box_size=PHOTUTILS_BOX,
        centroid_func=centroid_2dg_sky_sub,
    )
gauss_raw = np.full_like(input_xy, np.nan, dtype=float)
gauss_raw[in_frame] = np.column_stack([gx, gy])
gauss_xy, n_gauss_outbox = enforce_box_consistency(gauss_raw, input_xy, PHOTUTILS_BOX)
gauss_ok = np.isfinite(gauss_xy).all(axis=1)
print(f'Gauss outside-box drop  : {n_gauss_outbox}')
print(f'gauss_ok                : {gauss_ok.sum()} / {in_frame.sum()} attempted')


## 9. 2D Moffat centroids — bounded fit, gated on in_frame, post-validated

In [ ]:
def moffat_centroid_one(data, x0, y0, box=PHOTUTILS_BOX, fwhm_init=4.0, beta_init=3.0):
    cut = Cutout2D(data, (x0, y0), (box, box), mode='partial', fill_value=np.nan)
    arr = np.asarray(cut.data, dtype=float)
    if not np.isfinite(arr).all():
        return np.nan, np.nan
    yy, xx = np.mgrid[:arr.shape[0], :arr.shape[1]]
    bkg = float(np.nanmedian(arr))
    amp = float(np.nanmax(arr) - bkg)
    gamma_init = fwhm_init / (2.0 * np.sqrt(2.0 ** (1.0 / beta_init) - 1.0))
    moffat = Moffat2D(
        amplitude=amp,
        x_0=arr.shape[1] / 2.0,
        y_0=arr.shape[0] / 2.0,
        gamma=gamma_init,
        alpha=beta_init,
    )
    moffat.x_0.bounds       = (0.0, float(arr.shape[1] - 1))
    moffat.y_0.bounds       = (0.0, float(arr.shape[0] - 1))
    # gamma upper bound: 3*FWHM-equivalent. The previous bound (arr.shape[0])
    # let γ inflate to the full cutout width on noisy fits, producing
    # nonphysical "successful" fits for faint stars.
    moffat.gamma.bounds     = (0.1, 3.0 * fwhm_init)
    moffat.alpha.bounds     = (0.5, 10.0)
    moffat.amplitude.bounds = (0.0, None)
    model = moffat + Const2D(amplitude=bkg)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        try:
            fitted = TRFLSQFitter()(model, xx, yy, arr)
        except Exception:
            return np.nan, np.nan
    x_local = float(fitted[0].x_0.value)
    y_local = float(fitted[0].y_0.value)
    return cut.to_original_position((x_local, y_local))


# Sanity check on a synthetic noisy Moffat with a known center, using the same
# fwhm_init the production loop will use so the helper is exercised under
# production-like initial conditions.
rng = np.random.default_rng(0)
_yy, _xx = np.mgrid[:41, :41]
_truth = (20.4, 19.6)
_synth = Moffat2D(amplitude=1000.0, x_0=_truth[0], y_0=_truth[1],
                  gamma=2.0, alpha=3.0)(_xx, _yy)
_synth = _synth + 50.0 + rng.normal(0.0, 2.0, size=_synth.shape)
_xr, _yr = moffat_centroid_one(_synth, _truth[0], _truth[1],
                               box=15, fwhm_init=fwhm)
assert abs(_xr - _truth[0]) < 0.05 and abs(_yr - _truth[1]) < 0.05, \
    f'Moffat helper sanity check failed: got ({_xr:.3f}, {_yr:.3f})'
print(f'Moffat helper recovers synthetic center to ({_xr - _truth[0]:+.3f}, {_yr - _truth[1]:+.3f}) px')


In [ ]:
moffat_raw = np.full_like(input_xy, np.nan, dtype=float)
for i in tqdm(np.flatnonzero(in_frame), desc='Moffat fits'):
    moffat_raw[i] = moffat_centroid_one(
        balanced, input_xy[i, 0], input_xy[i, 1],
        box=PHOTUTILS_BOX, fwhm_init=fwhm,
    )
moffat_xy, n_moffat_outbox = enforce_box_consistency(moffat_raw, input_xy, PHOTUTILS_BOX)
moffat_ok = np.isfinite(moffat_xy).all(axis=1)
print(f'Moffat outside-box drop : {n_moffat_outbox} '
      f'(should be small — fit is bounded)')
print(f'moffat_ok               : {moffat_ok.sum()} / {in_frame.sum()} attempted')


## 10. Per-method success counts + hard correctness assertion

All three methods were attempted on exactly the same set of `in_frame.sum()` stars; the
ok counts differ only because each method has its own failure modes.

In [ ]:
print(f'total Gaia inputs        : {len(input_xy):4d}')
print(f'in-frame Gaia stars      : {in_frame.sum():4d}  '
      f'(every method attempted this exact set)')
print(f'Ballet ok (raw + box-ok) : {ballet_ok.sum():4d}  '
      f'(silent-fallback={n_silent}, outside-box={n_ballet_outbox})')
print(f'Gauss  ok (fit + box-ok) : {gauss_ok.sum():4d}  (outside-box={n_gauss_outbox})')
print(f'Moffat ok (fit + box-ok) : {moffat_ok.sum():4d}  (outside-box={n_moffat_outbox})')
print(f'all three ok             : {(ballet_ok & gauss_ok & moffat_ok).sum():4d}')

for name, arr, ok in [('Ballet', ballet_xy, ballet_ok),
                      ('Gauss',  gauss_xy,  gauss_ok),
                      ('Moffat', moffat_xy, moffat_ok)]:
    if ok.any():
        max_off = float(np.max(np.abs(arr[ok] - input_xy[ok])))
        assert max_off <= PHOTUTILS_BOX / 2 + 1e-6, \
            f'{name}: max |centroid - input| = {max_off:.2f} px > {PHOTUTILS_BOX/2}'
print(f'OK: all surviving centroids within {PHOTUTILS_BOX/2:.1f} px of their input')


## 11. Comparison table

In [ ]:
tab = Table()
tab['x_in']     = input_xy[:, 0]
tab['y_in']     = input_xy[:, 1]
tab['x_ballet'] = ballet_xy[:, 0]
tab['y_ballet'] = ballet_xy[:, 1]
tab['x_gauss']  = gauss_xy[:, 0]
tab['y_gauss']  = gauss_xy[:, 1]
tab['x_moffat'] = moffat_xy[:, 0]
tab['y_moffat'] = moffat_xy[:, 1]

def add_diff(tab, lo, hi):
    dx = tab[f'x_{hi}'] - tab[f'x_{lo}']
    dy = tab[f'y_{hi}'] - tab[f'y_{lo}']
    tab[f'dx_{hi}_minus_{lo}'] = dx
    tab[f'dy_{hi}_minus_{lo}'] = dy
    tab[f'dr_{hi}_minus_{lo}'] = np.hypot(dx, dy)

for lo, hi in [('gauss', 'ballet'), ('moffat', 'ballet'), ('moffat', 'gauss'),
               ('in', 'ballet'),    ('in', 'gauss'),     ('in', 'moffat')]:
    add_diff(tab, lo, hi)

tab[ballet_ok & gauss_ok & moffat_ok]


## 12. Summary statistics per comparison

In [ ]:
def summarize(label, dr_col, mask):
    arr = np.asarray(tab[dr_col])[mask]
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        print(f'{label:30s} N=   0  (no finite values)')
        return
    print(f'{label:30s} N={arr.size:4d}  median={np.median(arr):6.3f}  '
          f'MAD={median_absolute_deviation(arr):6.3f}  '
          f'p95={np.percentile(arr, 95):6.3f}  max={arr.max():6.3f}  px')

# Use one common mask for all three pairs so the rows are comparing the same
# population. Without this, Ballet-vs-Moffat (no Gauss drops) silently includes
# the faint stars that Gauss couldn't fit, making the rows non-comparable.
common_ok = ballet_ok & gauss_ok & moffat_ok

print(f'Pairwise centroid disagreements (N={common_ok.sum()}, common to all three):')
summarize('Ballet vs Gauss',   'dr_ballet_minus_gauss',  common_ok)
summarize('Ballet vs Moffat',  'dr_ballet_minus_moffat', common_ok)
summarize('Gauss  vs Moffat',  'dr_gauss_minus_moffat',  common_ok)

# NOTE: we deliberately do NOT report "<method> vs input" as accuracy.
# input_xy is gaia_photo_coords projected through compute_wcs(ref_coords[:15],
# ref.radecs[:15]), where ref_coords are skimage region centroids from
# detection.stars_detection. So |centroid - input| folds together
#   (a) the systematic offset between the method and skimage detection
#       (which set the WCS),
#   (b) WCS fit residuals from a 15-point solve, and
#   (c) real centroiding error.
# A method that happens to agree with skimage detection looks 'more accurate'
# by that metric regardless of which method actually centroids best.

if n_silent > 0:
    dr_prod = np.hypot(ballet_xy_prod[:, 0] - input_xy[:, 0],
                       ballet_xy_prod[:, 1] - input_xy[:, 1])
    arr = dr_prod[in_frame]
    arr = arr[np.isfinite(arr)]
    print()
    print(f'For reference, Ballet(prod) vs input over in-frame: '
          f'N={arr.size}, median={np.median(arr):.3f} px '
          f'(includes {n_silent} exact zeros from silent input-fallback)')


## 13. Plots

In [ ]:
pairs = [
    ('Ballet − Gauss',   'dr_ballet_minus_gauss',  'C0', ballet_ok & gauss_ok),
    ('Ballet − Moffat',  'dr_ballet_minus_moffat', 'C1', ballet_ok & moffat_ok),
    ('Gauss − Moffat',   'dr_gauss_minus_moffat',  'C2', gauss_ok  & moffat_ok),
]
fig, ax = plt.subplots(figsize=(7, 4))
for label, col, color, mask in pairs:
    arr = np.asarray(tab[col])[mask]
    arr = arr[np.isfinite(arr)]
    ax.hist(arr, bins=np.linspace(0, np.percentile(arr, 99), 60),
            histtype='stepfilled', alpha=0.4, color=color,
            label=f'{label} (N={arr.size})')
ax.set_xlabel('|Δr|  (pixels)')
ax.set_ylabel('# stars')
ax.set_title('Pairwise centroid disagreement')
ax.legend()
plt.tight_layout()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.3), sharex=True, sharey=True)
for ax, (label, col, color, mask) in zip(axes, pairs):
    dx = np.asarray(tab[col.replace('dr_', 'dx_')])[mask]
    dy = np.asarray(tab[col.replace('dr_', 'dy_')])[mask]
    m = np.isfinite(dx) & np.isfinite(dy)
    ax.scatter(dx[m], dy[m], s=6, alpha=0.4, color=color)
    th = np.linspace(0, 2 * np.pi, 200)
    ax.plot(np.cos(th), np.sin(th), '--', color='k', lw=0.6, label='1 px')
    ax.axhline(0, color='gray', lw=0.4); ax.axvline(0, color='gray', lw=0.4)
    ax.set_aspect('equal')
    ax.set_title(f'{label} (N={m.sum()})')
    ax.set_xlabel('Δx  (px)')
    ax.legend(loc='upper right', fontsize=8)
axes[0].set_ylabel('Δy  (px)')
lim = 2.0
axes[0].set_xlim(-lim, lim); axes[0].set_ylim(-lim, lim)
plt.tight_layout()


In [ ]:
spatial_mask = ballet_ok & gauss_ok
dr_bg = np.asarray(tab['dr_ballet_minus_gauss'])
m = np.isfinite(dr_bg) & spatial_mask
fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(input_xy[m, 0], input_xy[m, 1],
                c=np.clip(dr_bg[m], 0, np.percentile(dr_bg[m], 95)),
                s=10, cmap='viridis')
fig.colorbar(sc, ax=ax, label='|Δr|  Ballet − Gauss  (px, clipped p95)')
ax.set_xlabel('x (px)'); ax.set_ylabel('y (px)')
ax.set_title(f'Spatial map of Ballet vs Gauss disagreement (N={m.sum()})')
ax.set_aspect('equal')
plt.tight_layout()


In [ ]:
all_ok_mask = ballet_ok & gauss_ok & moffat_ok
all_ok_idx = np.flatnonzero(all_ok_mask)

# Rank by cutout peak rather than the value at the rounded-input pixel: input_xy
# is sub-pixel offset from the actual stellar core (that's what we're trying to
# centroid in the first place), so balanced[round(y), round(x)] can be ~half a
# pixel off the real peak and reorder the picks.
def cutout_peak(idx):
    cut = Cutout2D(balanced, (input_xy[idx, 0], input_xy[idx, 1]),
                   (PHOTUTILS_BOX, PHOTUTILS_BOX),
                   mode='partial', fill_value=np.nan)
    return float(np.nanmax(cut.data))

peaks = np.array([cutout_peak(i) for i in all_ok_idx])
order = np.argsort(peaks)
drbg = np.asarray(tab['dr_ballet_minus_gauss'])[all_ok_idx]
dr_order = np.argsort(drbg)

picks = {
    'brightest':       all_ok_idx[order[-1]],
    'faintest':        all_ok_idx[order[0]],
    'median bright':   all_ok_idx[order[len(order) // 2]],
    'smallest |Δr|':   all_ok_idx[dr_order[0]],
    'largest |Δr|':    all_ok_idx[dr_order[-1]],
}

fig, axes = plt.subplots(1, len(picks), figsize=(3 * len(picks), 3.2))
for ax, (label, idx) in zip(axes, picks.items()):
    cut = Cutout2D(balanced, (input_xy[idx, 0], input_xy[idx, 1]),
                   (PHOTUTILS_BOX, PHOTUTILS_BOX), mode='partial', fill_value=np.nan)
    ax.imshow(cut.data, origin='lower', cmap='gray',
              extent=(cut.bbox_original[1][0] - 0.5, cut.bbox_original[1][1] + 0.5,
                      cut.bbox_original[0][0] - 0.5, cut.bbox_original[0][1] + 0.5))
    ax.scatter(*input_xy[idx],  marker='o', s=80, edgecolor='cyan',  facecolor='none', label='input')
    ax.scatter(*ballet_xy[idx], marker='x', s=70, color='red',                       label='Ballet')
    ax.scatter(*gauss_xy[idx],  marker='+', s=90, color='lime',                      label='Gauss')
    ax.scatter(*moffat_xy[idx], marker='s', s=60, edgecolor='yellow', facecolor='none', label='Moffat')
    ax.set_title(f'{label}\nidx={idx}')
    ax.set_xticks([]); ax.set_yticks([])
axes[-1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8)
plt.tight_layout()


In [ ]:
good_tab = tab[all_ok_mask]
plt.figure()
plt.hist(good_tab["dx_ballet_minus_gauss"], bins=100, alpha=0.5, label='dx Ballet - Gauss')
plt.hist(good_tab["dy_ballet_minus_gauss"], bins=100, alpha=0.5, label='dy Ballet - Gauss')
plt.xlabel('Centroid offset (pixels)')
plt.ylabel('# stars')
plt.legend()
plt.grid()

## 14. Diagnostic experiments — why is Ballet's dx/dy not symmetric about zero?

Section 13 shows that `dx_ballet_minus_gauss` and `dy_ballet_minus_gauss` are not
symmetric about zero — Ballet has a directional bias relative to the analytic
methods. The Gauss/Moffat agreement (median |Δr| = 0.48 px, MAD = 0.43 px) is much
tighter than Ballet's disagreement with either of them (>1.2 px), so the bias lives
in Ballet, not in the analytic fits.

Hypotheses (ranked):

- **H1.** `bayer_balance_image` zeros the *global* checkerboard but each star has
  its own colour, so within each PSF cone there is a residual 2-pixel-period
  intensity pattern. Smooth analytic fits average it out; Ballet (a CNN trained on
  clean synthetic Moffats with no bayer structure) gets pulled by it. Bayer cells
  are 2 px on a side, which matches the observed >1 px disagreement scale.
- **H2.** Half-pixel coordinate-convention offset in `eloy.centroid` — the production
  formula `coords - 15/2 + cnn.centroid(cutouts)` uses 7.5 (geometric centre of a
  15×15 grid) where the central *pixel* is at index 7. ≤0.5 px in each axis, so
  too small to fully explain the bias on its own.
- **H3.** Asymmetric optical PSF (coma, astigmatism, guiding tail) the CNN handles
  differently than analytic fits.

Cells E1–E5 below are designed to discriminate among these. Predicted verdict matrix:

| E1 mean ≠ 0 | E2 phases separate | E4 clean recovers | Verdict        |
| ----------- | ------------------ | ----------------- | -------------- |
| yes         | yes                | yes               | H1 (bayer)     |
| yes (small) | no                 | yes               | H2 (offset)    |
| yes         | no                 | yes               | H3 (PSF)       |
| any         | no                 | no                | bug in eloy    |

### E1. Quantify the directional bias

Print mean / median / std of `(Δx, Δy)` for each pair on the common-OK set. The mean
is exactly what "not symmetric about zero" means; this puts a number on the bias.
Expected: large non-zero means on the four `*_ballet_*` rows, near-zero means on the
`moffat_minus_gauss` rows.

In [ ]:
print(f'Per-axis bias on the common-OK set (N={int(common_ok.sum())}):')
print()
for col in ['dx_ballet_minus_gauss',  'dy_ballet_minus_gauss',
            'dx_ballet_minus_moffat', 'dy_ballet_minus_moffat',
            'dx_gauss_minus_moffat',  'dy_gauss_minus_moffat']:
    arr = np.asarray(tab[col])[common_ok]
    arr = arr[np.isfinite(arr)]
    print(f'  {col:30s}  mean={arr.mean():+.3f}  median={np.median(arr):+.3f}  '
          f'std={arr.std():.3f}  px')

### E2. Bayer-phase scatter — H1 test

Plot Ballet−Moffat (Δx, Δy) coloured by the bayer phase of the star's central pixel.
H1 predicts the four phases sit in **different** quadrants and the union is offset only
because the phases are unequally populated. If the four clouds overlap (or all sit in
the same quadrant), H1 is wrong.

In [ ]:
phase = (np.round(input_xy[:, 0]).astype(int) % 2) * 2 \
      + (np.round(input_xy[:, 1]).astype(int) % 2)
m = common_ok
fig, ax = plt.subplots(figsize=(5.5, 5.2))
phase_names = ['(even x, even y)', '(even x, odd y)',
               '(odd x, even y)',  '(odd x, odd y)']
for p, name in enumerate(phase_names):
    sel = m & (phase == p)
    if sel.sum() == 0:
        continue
    ax.scatter(np.asarray(tab['dx_ballet_minus_moffat'])[sel],
               np.asarray(tab['dy_ballet_minus_moffat'])[sel],
               s=18, alpha=0.6,
               label=f'{name}  N={sel.sum()}')
ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.set_aspect('equal')
ax.set_xlabel('Δx (Ballet − Moffat) [px]')
ax.set_ylabel('Δy (Ballet − Moffat) [px]')
ax.set_title('Ballet − Moffat coloured by bayer phase of input pixel')
ax.legend(fontsize=8, loc='best')
ax.grid()
plt.tight_layout()

### E3. Magnitude trend

Plot `|Δr|` vs Gaia G. CNN training-distribution biases tend to grow at the bright
(saturated) and faint (noise-dominated) ends. A flat line argues for a uniform offset
(H2). `useful_gaia_mags` has one entry per row of `tab`, so the masks line up.

In [ ]:
m = common_ok
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(useful_gaia_mags[m], np.asarray(tab['dr_ballet_minus_moffat'])[m],
           s=10, alpha=0.5, label='Ballet − Moffat')
ax.scatter(useful_gaia_mags[m], np.asarray(tab['dr_gauss_minus_moffat'])[m],
           s=10, alpha=0.5, label='Gauss − Moffat')
ax.set_xlabel('Gaia G')
ax.set_ylabel('|Δr| (px)')
ax.set_title('Pairwise centroid disagreement vs source brightness')
ax.legend()
ax.grid()
plt.tight_layout()

### E4. Sanity check on a clean synthetic star

Feed Ballet the same `_synth` Moffat the Moffat sanity check uses (cell `f9953fb8`).
If Ballet recovers the synthetic centre to within a small fraction of a pixel, the CNN
is working correctly on its training distribution and the bias on the real image is
induced by image properties (H1 / H3). If it doesn't, the bug is in the CNN's coordinate
convention (H2) or in eloy itself.

In [ ]:
plt.figure()
plt.imshow(_synth[13:28, 13:28], origin='lower', cmap='gray')


In [ ]:
ballet_test = np.asarray(
    ballet_centroid(_synth.astype(float), np.array([_truth]), ref.cnn, nans=True)
)[0]
print(f'Ballet on clean synthetic Moffat:')
print(f'  truth     = {_truth}')
print(f'  recovered = ({ballet_test[0]:.3f}, {ballet_test[1]:.3f})')
print(f'  residual  = ({ballet_test[0] - _truth[0]:+.3f}, '
      f'{ballet_test[1] - _truth[1]:+.3f}) px')

In [ ]:
from eloy import utils
from astropy.nddata import Cutout2D

a_cutout = utils.cutout(_synth.astype(float), np.array([_truth]), (15, 15), fill_value=np.median(_synth.astype(float)))
real_cutout = Cutout2D(_synth.astype(float), _truth, (15, 15), mode='partial', fill_value=np.median(_synth.astype(float)))
bal_cen = ref.cnn.centroid(a_cutout)
print(bal_cen[0].shape)
print(f'a_cutout.shape: {a_cutout.shape}, \nbal_cen: {bal_cen}, \nreal_cutout.to_original_position(bal_cen[0]): {real_cutout.to_original_position(bal_cen[0])}, \ntruth: {_truth}, \nballet_test: {ballet_test}')

In [ ]:
_synth.shape

### E5. Spatial maps of Δx and Δy

Cell `833437c3` already maps `|Δr|`. Make two more, one each for `dx_ballet_minus_moffat`
and `dy_ballet_minus_moffat`, with a diverging colormap centred at zero. H1 should look
noisy at the pixel scale (bayer phase varies pixel-to-pixel); a smooth gradient across
the chip would point to H3 (off-axis aberrations).

In [ ]:
m = common_ok
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharey=True)
for ax, col in zip(axes, ['dx_ballet_minus_moffat', 'dy_ballet_minus_moffat']):
    vals = np.asarray(tab[col])[m]
    vmax = float(np.percentile(np.abs(vals), 95))
    sc = ax.scatter(input_xy[m, 0], input_xy[m, 1],
                    c=np.clip(vals, -vmax, vmax),
                    s=14, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    fig.colorbar(sc, ax=ax, label=f'{col} (px, clipped ±p95={vmax:.2f})')
    ax.set_xlabel('x (px)')
    ax.set_aspect('equal')
    ax.set_title(col)
axes[0].set_ylabel('y (px)')
plt.tight_layout()

## 15. Notes

_Fill in observations after running:_

- typical disagreement scale (median |Δr| in pixels) per pair
- how many fits each method had to drop for box-consistency
- whether disagreement is spatially structured
- behaviour at faint vs bright end
- which hypothesis (H1 / H2 / H3) the E1–E5 results support

In [ ]:
# All in-frame stars with Gaia G <= 13.5, with centroid locations overlaid.
# Same cutout idiom as the 5-star cell above, but the full bright population and
# only the three method markers (no input circle). NaN centroids don't plot, so
# stars where a method failed box-consistency simply show fewer markers.
BRIGHT_MAG = 13.5
NCOLS = 6

# Bright + in-frame stars, then sort that subset brightest -> faintest.
# flatnonzero gives integer row indices into the full per-star arrays
# (input_xy / ballet_xy / gauss_xy / moffat_xy all share one row per Gaia star).
bright = (useful_gaia_mags <= BRIGHT_MAG) & in_frame
bright_idx = np.flatnonzero(bright)
bright_idx = bright_idx[np.argsort(useful_gaia_mags[bright_idx])]  # brightest first
print(f'{bright_idx.size} in-frame stars with G <= {BRIGHT_MAG}')

# One panel per star; ceil so a partial final row still gets axes. squeeze=False
# keeps `axes` 2D even when nrows or NCOLS is 1, so .ravel() is always valid.
nrows = int(np.ceil(bright_idx.size / NCOLS))
fig, axes = plt.subplots(nrows, NCOLS, figsize=(3 * NCOLS, 3 * nrows), squeeze=False)
axes = axes.ravel()

for ax, idx in zip(axes, bright_idx):
    # PHOTUTILS_BOX square around the star's input pixel. mode='partial' lets the
    # box hang off the image edge (filled with NaN) instead of being clipped small.
    cut = Cutout2D(balanced, (input_xy[idx, 0], input_xy[idx, 1]),
                   (PHOTUTILS_BOX, PHOTUTILS_BOX), mode='partial', fill_value=np.nan)
    # bbox_original is ((y0, y1), (x0, x1)) in FULL-image pixels. Drawing the cutout
    # with extent set to those bounds (-/+0.5 puts edges on pixel borders) renders it
    # in original-image coords, so the full-image centroid markers below land right.
    ax.imshow(cut.data, origin='lower', cmap='gray',
              extent=(cut.bbox_original[1][0] - 0.5, cut.bbox_original[1][1] + 0.5,
                      cut.bbox_original[0][0] - 0.5, cut.bbox_original[0][1] + 0.5))
    # *xy unpacks the (x, y) centroid as scatter's two positional args; a NaN
    # centroid (method failed / dropped by box-consistency) just draws nothing.
    ax.scatter(*ballet_xy[idx], marker='x', s=70, color='red',                       label='Ballet')
    ax.scatter(*gauss_xy[idx],  marker='+', s=90, color='lime',                      label='Gauss')
    ax.scatter(*moffat_xy[idx], marker='s', s=60, edgecolor='yellow', facecolor='none', label='Moffat')
    ax.set_title(f'G={useful_gaia_mags[idx]:.2f}\nidx={idx}', fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])

# Hide the unused trailing panels in the last row.
for ax in axes[bright_idx.size:]:
    ax.axis('off')

# Single shared legend from the first populated panel.
axes[0].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8)
plt.tight_layout()


## 16. Phase 1 re-measure (post coordinate-fix in `eloy.ballet_centroid`)

The earlier >1 px Ballet bias was dominated by a cutout→image coordinate bug (H2), now
fixed: `ballet_centroid` back-transforms via `Cutout2D.to_original_position`. With that
fixed, the question is whether faint-star centroids still need work. §16 bins the real-image
disagreement by Gaia G; §17 measures ground-truth accuracy vs SNR on synthetics. Compare E1's
per-axis Ballet bias below against the pre-fix values in `ballet_centroid_bias_diagnosis.md`
(mean Δ ≈ +0.1 / +0.9 px) — it should have collapsed toward zero.

In [ ]:
# Per-magnitude disagreement, post coordinate-fix. Bin the common-OK stars by Gaia G and
# report median/MAD |Δr| per bin. dr_gauss_minus_moffat is the control: if Ballet tracks
# it across all bins there is no faint-star problem. useful_gaia_mags is one-per-tab-row.
m        = common_ok                       # stars where every method gave a valid centroid
mags_ok  = np.asarray(useful_gaia_mags)[m] # Gaia G for those same rows (parallel to `tab`)
# Three method-pair |Δr| series, each subset to the common-OK rows so they cover the
# same stars. Gauss-Moffat is the control: two analytic fits on identical data, so its
# spread is the method floor that Ballet's disagreement should be compared against.
cols = {
    'Ballet-Moffat': np.asarray(tab['dr_ballet_minus_moffat'])[m],
    'Ballet-Gauss':  np.asarray(tab['dr_ballet_minus_gauss'])[m],
    'Gauss-Moffat':  np.asarray(tab['dr_gauss_minus_moffat'])[m],   # control
}
# 0.5-mag bin edges spanning the data; `centers` are the bin midpoints for plotting.
edges   = np.arange(np.floor(mags_ok.min()), np.ceil(mags_ok.max()) + 0.5, 0.5)
centers = 0.5 * (edges[:-1] + edges[1:])

# Header row: right-justified column titles, one per method pair.
print(f'Per-magnitude median |Δr| / MAD (N={m.sum()} common-OK stars), 0.5-mag bins:')
print(f'{"G bin":>11s} {"N":>4s} ' + ' '.join(f'{k:>20s}' for k in cols))
med_by_pair = {k: [] for k in cols}        # collect per-bin medians to plot afterwards
for lo, hi in zip(edges[:-1], edges[1:]):
    sel = (mags_ok >= lo) & (mags_ok < hi) # stars in this magnitude bin
    parts = []
    for k, v in cols.items():
        vv = v[sel]; vv = vv[np.isfinite(vv)]                       # drop NaNs in this bin
        med_by_pair[k].append(np.median(vv) if vv.size else np.nan) # median for the plot
        # Formatted "median/MAD" cell; '--' when the bin is empty for this series.
        parts.append(f'{np.median(vv):6.3f}/{median_absolute_deviation(vv):5.3f}'
                     if vv.size else '       --')
    if sel.sum():                          # only print bins that actually contain stars
        print(f'{lo:5.1f}-{hi:4.1f} {sel.sum():4d} ' + ' '.join(f'{p:>20s}' for p in parts))

# Plot the per-bin median |Δr| curves; Ballet sitting on the Gauss-Moffat control = no
# faint-star problem.
fig, ax = plt.subplots(figsize=(7, 4))
for k, med in med_by_pair.items():
    ax.plot(centers, med, 'o-', label=k)
ax.set_xlabel('Gaia G'); ax.set_ylabel('median |Δr| (px)')
ax.set_title('Per-magnitude centroid disagreement (Gauss-Moffat = control)')
ax.legend(); ax.grid()
plt.tight_layout()


### 17. Phase 1: ground-truth accuracy vs SNR (synthetic)

Pairwise agreement (§16) is not accuracy. Here we centroid synthetic Moffats at **known**
sub-pixel centers across a sweep of peak SNR, with a realistic sky pedestal + Poisson +
read noise — the regime the Ballet CNN never saw in training (amplitude fixed at 1,
background 0, uniform noise ≤ 0.1). The `bayer` variant imposes a per-channel gain and runs
the real `bayer_balance_image` to isolate the checkerboard's cost. **Decision gate:** if
Ballet's faint-end (low-SNR) error rises markedly faster than Gauss/Moffat, Phase 2
(realistic-noise retrain) is warranted; if comparable, retraining is not.

In [ ]:
# Synthetic SNR sweep: ground-truth recovery error vs brightness.
# The mag-binned cells measure method *agreement*, which is not accuracy. Here we centroid
# Moffats with KNOWN sub-pixel centers over a range of peak SNR. Two variants per SNR: a plain
# star (sky + Poisson + read noise) and the same star with a per-Bayer-channel gain imposed then
# pushed through the exact production bayer_balance_image -- isolating whether the residual
# checkerboard costs Ballet accuracy at the faint end specifically.
#
# This cell defines the SHARED sweep machinery -- sweep_truth() / plot_truth() -- and runs the
# §17 case (old Ballet, ref.cnn). §18b and §19b reuse the same two helpers for the retrained
# models, so all three sections share one implementation. §17 is just the single-model call.
from astropy.modeling.models import Moffat2D as _Moffat2D_syn

rng_sweep   = np.random.default_rng(42)   # reseeded per sweep_truth() call; see there
FRAME       = 41                          # synthetic cutout is FRAME x FRAME pixels
BETA_SYN    = 3.0                          # Moffat beta (alpha) shape parameter
# Convert the image FWHM to a Moffat gamma for this beta: FWHM = 2*gamma*sqrt(2**(1/beta)-1),
# so this gamma reproduces the real frame's PSF width in the synthetic stars.
GAMMA_SYN   = fwhm / (2.0 * np.sqrt(2.0 ** (1.0 / BETA_SYN) - 1.0))   # match image FWHM
SKY         = 100.0                        # flat background level (counts)
READ_NOISE  = 5.0                          # Gaussian read noise std (counts)
NOISE_FLOOR = np.sqrt(SKY + READ_NOISE ** 2)        # background-limited noise std
SNR_GRID    = np.array([2, 3, 5, 8, 12, 20, 40, 80, 160], dtype=float)  # peak SNR points
N_PER       = 50                           # random star realizations per SNR per variant
BAYER_GAINS = (1.00, 1.35, 1.35, 0.80)              # R, G, G, B relative response
variants    = ['plain', 'bayer']           # without / with the Bayer-channel imbalance
_yy_s, _xx_s = np.mgrid[:FRAME, :FRAME]    # pixel coordinate grids for evaluating the model

def _make_star(amp, cx, cy, bayer=False):
    # Noise-free Moffat of peak `amp` centered at (cx, cy), plus the flat sky.
    clean = _Moffat2D_syn(amplitude=amp, x_0=cx, y_0=cy,
                          gamma=GAMMA_SYN, alpha=BETA_SYN)(_xx_s, _yy_s) + SKY
    if bayer:
        # Impose the per-CFA-channel gain on the 2x2 Bayer mosaic (R, G, G, B), i.e.
        # before noise -- this is the channel imbalance the pipeline later rebalances.
        clean = clean.copy()
        clean[0::2, 0::2] *= BAYER_GAINS[0]; clean[0::2, 1::2] *= BAYER_GAINS[1]
        clean[1::2, 0::2] *= BAYER_GAINS[2]; clean[1::2, 1::2] *= BAYER_GAINS[3]
    # Photon (Poisson) noise on the clipped signal + Gaussian read noise.
    noisy = (rng_sweep.poisson(np.clip(clean, 0, None)).astype(float)
             + rng_sweep.normal(0.0, READ_NOISE, clean.shape))
    if bayer:
        bayer_balance_image(noisy)   # in place, exactly as the pipeline does
    return noisy

def sweep_truth(models, seed=42, desc=''):
    """Ground-truth accuracy sweep, generalized over a set of Ballet models.

    models : ordered dict {label: cnn}. Each Ballet model is centroided alongside the
    analytic Gauss and Moffat baselines on the SAME synthetic stars. Reseeding rng_sweep
    here and reusing _make_star in the original §17 draw order means every call -- and the
    single-model §17 call -- sees identical stars, so old/realistic/bayer are directly
    comparable. Gauss/Moffat/Ballet draw no rng, so model order doesn't perturb the stream.
    Returns errs[variant][label][snr] -> list of |Δr| recovery errors (px).
    """
    global rng_sweep
    rng_sweep = np.random.default_rng(seed)             # fresh, deterministic star set
    labels = list(models) + ['Gauss', 'Moffat']         # plotting/column order
    errs = {v: {lab: {s: [] for s in SNR_GRID} for lab in labels} for v in variants}
    for v in variants:
        for snr in tqdm(SNR_GRID, desc=f'{desc} ({v})' if desc else v):
            amp = snr * NOISE_FLOOR                      # peak amplitude for this SNR
            for _ in range(N_PER):
                # Random sub-pixel truth near frame center (+/-1.5 px) so we test phase too.
                cx = FRAME / 2 + rng_sweep.uniform(-1.5, 1.5)
                cy = FRAME / 2 + rng_sweep.uniform(-1.5, 1.5)
                noisy = _make_star(amp, cx, cy, bayer=(v == 'bayer'))
                sx, sy = round(cx), round(cy)            # shared integer search seed
                rec = {}                                 # recovered (x, y) per label
                # Each requested Ballet model (nans=True -> failure returns NaN, not raise).
                for lab, cnn in models.items():
                    rec[lab] = ballet_centroid(noisy, np.array([[float(sx), float(sy)]]),
                                               cnn, nans=True)[0]
                try:
                    # photutils 2D-Gaussian; suppress fit-convergence warnings.
                    with warnings.catch_warnings():
                        warnings.simplefilter('ignore')
                        gx, gy = centroid_sources(noisy, np.array([float(sx)]), np.array([float(sy)]),
                                                  box_size=PHOTUTILS_BOX,
                                                  centroid_func=centroid_2dg_sky_sub)
                    rec['Gauss'] = (float(gx[0]), float(gy[0]))
                except Exception:
                    rec['Gauss'] = (np.nan, np.nan)      # failed fit counts as a miss
                # Bounded 2D-Moffat fit (same helper used on the real frame).
                rec['Moffat'] = moffat_centroid_one(noisy, sx, sy, box=PHOTUTILS_BOX, fwhm_init=fwhm)
                for lab, (rx, ry) in rec.items():
                    e = np.hypot(rx - cx, ry - cy)       # recovery error vs known truth
                    if np.isfinite(e) and e <= PHOTUTILS_BOX / 2:   # box-consistency rule
                        errs[v][lab][snr].append(e)
    return errs

def plot_truth(errs_d, suptitle):
    """Two-panel (plain/bayer) log-x median-error plot + text table for a sweep_truth result."""
    labels = list(next(iter(errs_d.values())).keys())   # label order taken from the errs dict
    # One panel per variant; shared y so plain vs bayer accuracy is directly comparable.
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
    for ax, v in zip(axes, variants):
        for lab in labels:
            # Median error per SNR (NaN where every realization was rejected).
            med = [np.median(errs_d[v][lab][s]) if errs_d[v][lab][s] else np.nan for s in SNR_GRID]
            nmin = min(len(errs_d[v][lab][s]) for s in SNR_GRID)   # worst-case surviving count
            ax.plot(SNR_GRID, med, 'o-', label=f'{lab} (min N={nmin})')
        ax.set_xscale('log')
        ax.set_xlabel('peak SNR  (amp / background-noise floor)')
        ax.set_title(f'{v} synthetic stars')
        ax.grid(which='both', alpha=0.3)
        ax.legend()
    axes[0].set_ylabel('median recovery error |Δr| vs truth (px)')
    fig.suptitle(suptitle)
    plt.tight_layout()

    # Same medians as a text table; label column widened to fit the longest label.
    w = max(len(lab) for lab in labels)
    print('median recovery error (px) by SNR:')
    for v in variants:
        print(f'  [{v}]')
        print(f'    {"SNR":<{w}} ' + '  '.join(f'{int(s):>6d}' for s in SNR_GRID))
        for lab in labels:
            row = '  '.join((f'{np.median(errs_d[v][lab][s]):6.3f}' if errs_d[v][lab][s] else '    --')
                            for s in SNR_GRID)
            print(f'    {lab:<{w}} {row}')

# §17 itself is the single-model case: just the old Ballet (ref.cnn). Same output as before.
errs = sweep_truth({'Ballet': ref.cnn}, desc='SNR sweep')
plot_truth(errs, 'Ground-truth centroid accuracy vs SNR  (lower is better)')

## 18. New-weights Ballet vs old Ballet vs Moffat vs Gaussian

Same per-magnitude framing as §16, with the realistic-noise retrain
(`ballet_realistic_15x15.npz`, Phase 2) added as **Ballet(new)**. It is built as a *separate*
`Ballet` object so the deployed model (`ref.cnn`) and the existing `ballet_xy` / `ballet_ok`
fits are untouched. These weights are a local CPU retrain — **not** the pipeline default, which
still loads the HuggingFace `centroid_15x15.npz`. Moffat is the shared reference and
`Gauss−Moffat` is the control; `Ballet(new)−Ballet(old)` shows how far the retrain moved the
centroids. Expect `Ballet(new)−Moffat` ≤ `Ballet(old)−Moffat` in the faint bins (G ≳ 12.5),
mirroring the §17 synthetic result.

In [ ]:
# §18 prerequisite: run Ballet with the realistic-noise retrain weights.
# Build a SEPARATE Ballet object so ref.cnn / ballet_xy / ballet_ok stay untouched.
# Loads a local .npz (no network) and mirrors §7's gating + box-consistency exactly.
NEW_WEIGHTS = '/Users/mattcraig/development/astronomy/eloy/ballet_realistic_15x15.npz'
ballet_new_cnn = Ballet(NEW_WEIGHTS)       # independent model; does not touch ref.cnn

# Centroid only the in-frame stars (same gate as §7); nans=True -> silent failures
# come back as NaN rather than raising.
ballet_new_inframe = np.asarray(
    ballet_centroid(balanced, input_xy[in_frame], ballet_new_cnn, nans=True)
)
# Scatter the in-frame results back into a full-length array (NaN elsewhere), exactly
# as §7 does, so the rows stay aligned with every other per-star array and with `tab`.
ballet_new_raw = np.full_like(input_xy, np.nan, dtype=float)
ballet_new_raw[in_frame] = ballet_new_inframe
n_new_silent = int(np.sum(np.isnan(ballet_new_inframe).any(axis=1)))   # CNN returned NaN
# Reject centroids that wandered outside the photometry box (same rule as the old model).
ballet_new_xy, n_new_outbox = enforce_box_consistency(ballet_new_raw, input_xy, PHOTUTILS_BOX)
ballet_new_ok = np.isfinite(ballet_new_xy).all(axis=1)                 # survived both gates
print(f'Ballet(new) NaN from CNN     : {n_new_silent} (of {in_frame.sum()} in-frame attempts)')
print(f'Ballet(new) outside-box drop : {n_new_outbox}')
print(f'ballet_new_ok                : {ballet_new_ok.sum()} / {in_frame.sum()} attempted')
print(f'(old) ballet_ok              : {ballet_ok.sum()} / {in_frame.sum()}  -- unchanged')

# Add new-weights columns + pairwise diffs (reuse add_diff from §11). Only ADD columns;
# the existing x_ballet/y_ballet and their diffs are left exactly as built in §11.
tab['x_balletnew'] = ballet_new_xy[:, 0]
tab['y_balletnew'] = ballet_new_xy[:, 1]
# Build dr_*_minus_balletnew columns vs Moffat, Gauss, and the old Ballet.
for lo, hi in [('moffat', 'balletnew'), ('gauss', 'balletnew'), ('ballet', 'balletnew')]:
    add_diff(tab, lo, hi)


In [ ]:
# §18: per-magnitude comparison, all referenced to Moffat (same framing as §16), with the
# new-weights Ballet added. Common mask requires every method OK so the series compare the
# same stars. Ballet(new)-Ballet(old) shows how far the retrain moved the centroids.
m4    = ballet_ok & ballet_new_ok & gauss_ok & moffat_ok   # rows valid for ALL four methods
mags4 = np.asarray(useful_gaia_mags)[m4]                   # Gaia G for those rows
# Four |Δr| series on the common-OK rows. Gauss-Moffat is the method-floor control;
# Ballet(new)-Ballet(old) measures how much the retrain shifted the centroids.
series = {
    'Ballet(old)-Moffat':      np.asarray(tab['dr_ballet_minus_moffat'])[m4],
    'Ballet(new)-Moffat':      np.asarray(tab['dr_balletnew_minus_moffat'])[m4],
    'Gauss-Moffat':            np.asarray(tab['dr_gauss_minus_moffat'])[m4],   # control
    'Ballet(new)-Ballet(old)': np.asarray(tab['dr_balletnew_minus_ballet'])[m4],
}

# Whole-sample summary before the per-bin breakdown.
print(f'Overall median |Δr| over common-OK (N={m4.sum()}):')
for k, v in series.items():
    vv = v[np.isfinite(v)]
    print(f'  {k:24s} median={np.median(vv):6.3f}  MAD={median_absolute_deviation(vv):6.3f}  px')

# 0.5-mag bins over the common-OK magnitudes (same idiom as §16).
edges4   = np.arange(np.floor(mags4.min()), np.ceil(mags4.max()) + 0.5, 0.5)
centers4 = 0.5 * (edges4[:-1] + edges4[1:])
print(f'\nPer-magnitude median |Δr| / MAD (N={m4.sum()}), 0.5-mag bins:')
print(f'{"G bin":>11s} {"N":>4s} ' + ' '.join(f'{k:>24s}' for k in series))
med_by = {k: [] for k in series}            # per-bin medians collected for the plot
for lo, hi in zip(edges4[:-1], edges4[1:]):
    sel = (mags4 >= lo) & (mags4 < hi)      # stars in this magnitude bin
    parts = []
    for k, v in series.items():
        vv = v[sel]; vv = vv[np.isfinite(vv)]
        med_by[k].append(np.median(vv) if vv.size else np.nan)
        parts.append(f'{np.median(vv):6.3f}/{median_absolute_deviation(vv):5.3f}'
                     if vv.size else '       --')
    if sel.sum():
        print(f'{lo:5.1f}-{hi:4.1f} {sel.sum():4d} ' + ' '.join(f'{p:>24s}' for p in parts))

# Per-magnitude median |Δr| curves: old vs new Ballet against the Moffat reference.
fig, ax = plt.subplots(figsize=(8, 4.5))
for k, med in med_by.items():
    ax.plot(centers4, med, 'o-', label=k)
ax.set_xlabel('Gaia G'); ax.set_ylabel('median |Δr| (px)')
ax.set_title('Per-magnitude comparison: old vs new Ballet (Moffat-referenced)')
ax.legend(); ax.grid()
plt.tight_layout()


### 18b. Ground-truth accuracy — realistic-noise Ballet

Same synthetic SNR sweep as §17, but comparing the §18 realistic-noise retrain
(`ballet_new_cnn`) against the old Ballet and the analytic Gauss/Moffat baselines on
**identical** synthetic stars (`sweep_truth` reseeds to the same star set). §18's per-magnitude
plot measures only *agreement* with Moffat; this measures **accuracy vs known truth**, so it
shows whether the retrain actually moved recovery error — especially at the faint end (SNR ≲ 8).

In [ ]:
# §18 ground truth: realistic-noise retrain vs old Ballet vs Gauss/Moffat, accuracy vs truth.
# Reuses sweep_truth/plot_truth from §17; same seed -> same synthetic stars as §17, so the
# Ballet(old) curve here reproduces §17's and Ballet(realistic) is directly comparable to it.
errs_18b = sweep_truth(
    {'Ballet(old)': ref.cnn, 'Ballet(realistic)': ballet_new_cnn},
    desc='ground truth (realistic)',
)
plot_truth(errs_18b, 'Ground-truth accuracy vs SNR: realistic-noise Ballet  (lower is better)')

## 19. Phase 3: bayer-aware fine-tune vs old & realistic Ballet

Adds the Phase-3 bayer-aware fine-tune (`ballet_realistic_bayer_15x15.npz`) as a third model,
built as a separate `Ballet` object (the old `ballet_*` and §18's realistic `ballet_new_*` are
untouched). On synthetics (§17-style) it is the best CNN on bayer-balanced data — recovering the
bayer performance the noise-only retrain lost and improving the mid-SNR range — while a ~0.06 px
CNN floor remains at the very bright end (vs Moffat's ~0.02). Here we check the real-image bins,
especially bright (G < 12). All series are referenced to Moffat; `Gauss−Moffat` is the control.
None of these are the pipeline default (still HF `centroid_15x15.npz`).

In [ ]:
# §19 prerequisite: run Ballet with the Phase-3 bayer-aware fine-tune weights.
# Separate Ballet object; the old ballet_* and §18's realistic ballet_new_* are untouched.
BAYER_WEIGHTS = '/Users/mattcraig/development/astronomy/eloy/ballet_realistic_bayer_15x15.npz'
ballet_bayer_cnn = Ballet(BAYER_WEIGHTS)   # third independent model

# Same gate-then-scatter-then-box-check pipeline as §7/§18 (see cell 55 for the steps).
ballet_bayer_inframe = np.asarray(
    ballet_centroid(balanced, input_xy[in_frame], ballet_bayer_cnn, nans=True)
)
ballet_bayer_raw = np.full_like(input_xy, np.nan, dtype=float)
ballet_bayer_raw[in_frame] = ballet_bayer_inframe
ballet_bayer_xy, n_bayer_outbox = enforce_box_consistency(ballet_bayer_raw, input_xy, PHOTUTILS_BOX)
ballet_bayer_ok = np.isfinite(ballet_bayer_xy).all(axis=1)
print(f'Ballet(bayer) outside-box drop : {n_bayer_outbox}')
print(f'ballet_bayer_ok                : {ballet_bayer_ok.sum()} / {in_frame.sum()} attempted')

# Add bayer-model columns + diffs vs Moffat, old Ballet, and the realistic (new) Ballet.
tab['x_balletbayer'] = ballet_bayer_xy[:, 0]
tab['y_balletbayer'] = ballet_bayer_xy[:, 1]
for lo, hi in [('moffat', 'balletbayer'), ('ballet', 'balletbayer'), ('balletnew', 'balletbayer')]:
    add_diff(tab, lo, hi)


In [ ]:
# §19: per-magnitude comparison of all three Ballet models vs Moffat (Gauss-Moffat control).
# old = HF pipeline default (ballet_xy); realistic = Phase-2 (ballet_new_xy from §18);
# bayer = Phase-3 bayer-aware (ballet_bayer_xy). Common mask requires every method OK.
m5    = ballet_ok & ballet_new_ok & ballet_bayer_ok & gauss_ok & moffat_ok  # all 5 valid
mags5 = np.asarray(useful_gaia_mags)[m5]
# All three Ballet models referenced to Moffat, plus Gauss-Moffat as the method floor.
series = {
    'Ballet(old)-Moffat':       np.asarray(tab['dr_ballet_minus_moffat'])[m5],
    'Ballet(realistic)-Moffat': np.asarray(tab['dr_balletnew_minus_moffat'])[m5],
    'Ballet(bayer)-Moffat':     np.asarray(tab['dr_balletbayer_minus_moffat'])[m5],
    'Gauss-Moffat':             np.asarray(tab['dr_gauss_minus_moffat'])[m5],   # control
}

# Whole-sample summary before the per-bin breakdown.
print(f'Overall median |Δr| over common-OK (N={m5.sum()}):')
for k, v in series.items():
    vv = v[np.isfinite(v)]
    print(f'  {k:26s} median={np.median(vv):6.3f}  MAD={median_absolute_deviation(vv):6.3f}  px')

# 0.5-mag bins; this table prints medians only (no MAD) to keep the wider table readable.
edges5   = np.arange(np.floor(mags5.min()), np.ceil(mags5.max()) + 0.5, 0.5)
centers5 = 0.5 * (edges5[:-1] + edges5[1:])
print(f'\nPer-magnitude median |Δr| (N={m5.sum()}), 0.5-mag bins:')
print(f'{"G bin":>11s} {"N":>4s} ' + ' '.join(f'{k:>26s}' for k in series))
med_by = {k: [] for k in series}
for lo, hi in zip(edges5[:-1], edges5[1:]):
    sel = (mags5 >= lo) & (mags5 < hi)
    parts = []
    for k, v in series.items():
        vv = v[sel]; vv = vv[np.isfinite(vv)]
        med_by[k].append(np.median(vv) if vv.size else np.nan)
        parts.append(f'{np.median(vv):6.3f}' if vv.size else '    --')
    if sel.sum():
        print(f'{lo:5.1f}-{hi:4.1f} {sel.sum():4d} ' + ' '.join(f'{p:>26s}' for p in parts))

# Per-magnitude curves for all three Ballet models against the Moffat reference.
fig, ax = plt.subplots(figsize=(8, 4.5))
for k, med in med_by.items():
    ax.plot(centers5, med, 'o-', label=k)
ax.set_xlabel('Gaia G'); ax.set_ylabel('median |Δr| (px)')
ax.set_title('Per-magnitude: old vs realistic vs bayer Ballet (Moffat-referenced)')
ax.legend(); ax.grid()
plt.tight_layout()


### 19b. Ground-truth accuracy — bayer-aware Ballet

Synthetic SNR sweep comparing all three Ballet models — old (`ref.cnn`), realistic
(`ballet_new_cnn`), and the §19 bayer-aware fine-tune (`ballet_bayer_cnn`) — against the
Gauss/Moffat baselines on identical stars. The **bayer panel** is the decisive one: it is exactly
the residual-checkerboard regime the Phase-3 fine-tune targets, so this is where the bayer model
should close the gap to the analytic floor that old Ballet never reached.

In [ ]:
# §19 ground truth: all three Ballet models (old / realistic / bayer-aware) vs Gauss/Moffat,
# accuracy vs truth. Same shared sweep on the same seeded stars; the bayer panel is the one
# that matters for the Phase-3 fine-tune.
errs_19b = sweep_truth(
    {'Ballet(old)': ref.cnn,
     'Ballet(realistic)': ballet_new_cnn,
     'Ballet(bayer)': ballet_bayer_cnn},
    desc='ground truth (bayer)',
)
plot_truth(errs_19b, 'Ground-truth accuracy vs SNR: old vs realistic vs bayer Ballet  (lower is better)')